Se importa el archivo .csv con los datos ya limpios y unicamente las variables seleccionadas 

In [1]:
import pandas as pd
import numpy as np

# Cargar el output de los datos limpios
df = pd.read_csv('../outputs/01_datos_limpios.csv')

print(f"Datos cargados: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Tarimas distintas: {df['ARMA_TARIMA'].nunique():,}")

Datos cargados: 27,794 filas × 16 columnas
Tarimas distintas: 9,817


Se busca que tipo de embalaje tiene cada cinta

In [2]:
# Extraer tipo de embalaje desde DESCRIPCION_EMBALAJE
def clasificar_embalaje(descripcion):
    if pd.isna(descripcion):
        return 'DESCONOCIDO'
    descripcion = str(descripcion).upper()
    if 'OJO HORIZONTAL' in descripcion:
        return 'OJO_HORIZONTAL'
    elif 'OJO VERTICAL' in descripcion:
        return 'OJO_VERTICAL'
    else:
        return 'OTRO'

df['TIPO_EMBALAJE'] = df['DESCRIPCION_EMBALAJE'].apply(clasificar_embalaje)

print("Distribución de tipos de embalaje:")
print(df['TIPO_EMBALAJE'].value_counts())

Distribución de tipos de embalaje:
TIPO_EMBALAJE
OJO_HORIZONTAL    15586
OJO_VERTICAL      12208
Name: count, dtype: int64


Se crea el siguiente DataFrame añadiendo variables para que resulte más sencillo para el modelo ML identificar si se viola alguna de las siguientes 7 reglas, que desenbocaría en el desarme de una tarima:


1. Mismo material padre: Todas las cintas de una tarima deben tener el mismo valor en `MATERIAL_PADRE`. 


2. Mismo consignatario: Todas las cintas de una tarima deben tener el mismo valor en `CONSIGNATARIO`.


3. Peso máximo no superado: La suma de `PESO_NETO_DESP` de todas las cintas en la tarima no debe superar `PESO_MAX_DESPACHO_HIJO`. Este valor frecuentemente es `NOA`, por lo que se tiene que obtener de los datos


4. Número máximo de piezas no superado: La cantidad de cintas en la tarima no debe superar `PIEZAS_POR_PAQ_HIJO`. Este valor usualmente es `NOA` por lo que se tiene que obtener de los datos


5. Diferencia de diámetro externo menor a 3 pulgadas: El diámetro interno de todas las cintas es siempre 508 mm. El diámetro externo se calcula a partir de `LARGO_HIJO` (largo en metros, se convierte a mm multiplicando por 1,000) y `ESPESOR_HIJO` (espesor en mm) usando la siguiente fórmula D_externo = √(D_INT² + 4 × (LARGO_HIJO × 1000) × ESPESOR_HIJO / π)
La diferencia entre el diámetro externo mayor y el menor dentro de una misma tarima no debe superar 3 pulgadas (~76.2 mm).


6. Restricción por tipo de embalaje OJO HORIZONTAL (esta se encuentra en `DESCRIPCION_EMBALAJE` cuando contiene "OJO HORIZONTAL" en el dato, Por ejemplo `"EC52 - CINTA / PAPEL / OJO HORIZONTAL / TERRESTRE"`): El ancho máximo del atado debe ser menor a 1500 mm, excepto en `SLT4_CHU` donde el límite es 1700 mm.


7. Restricción por tipo de embalaje OJO VERTICAL (se encuentra en `DESCRIPCION_EMBALAJE` cuando contiene "OJO VERTICAL" escrito, Por ejemplo `"EC85 - CINTA / PAPEL / OJO VERTICAL / TERRESTRE"`): La altura de la tarima debe ser menor a 250 mm.


Se genera el dataframe y sus variables de la siguiente manera:


Se colapsa el dataframe de contener cada fila una cinta a que cada fila represente una tarima:

- DESARMA_TARIMA: representa si se desarmó la tarima o no (1 si sí se desarmó, 0 si no )

- n_cintas: Cuantas cintas hay por tarima 

- peso_total: peso total de cada tarima (suma de los pesos de todas las cintas que tiene la tarima)

- n_materiales_padre: cuenta si en una tarima se mezclaron varias cintas de distintos materiales padre

- n_consignatarios: cuenta si en una tarima se mezclaron varias cintas que van a diferente consignatario

- diff_diametro: Es la diferencia entre el diametro externo mas grande y el diametro externo mas pequeño de las cintas que pertenecen a una misma tarima

- tipo_embalaje: tipo de embalaje que tiene cada tarima (OJO HORIZONTAL/OJO VERTICAL)

- ancho_max: ancho de la cinta más grande

- n_numeros_parte: cuenta cuántos NUMERO_PARTE_HIJO hay en una tarima

- D_UBICACION: Muestra en qué planta se fabricaron y empaquetaron las cintas

In [3]:
vista_tarimas = df.groupby('ARMA_TARIMA').agg(
    # Variable objetivo
    DESARMA_TARIMA=('DESARMA_TARIMA', 'max'),
    
    # Regla 4 — número de cintas
    n_cintas=('PESO_NETO_DESP', 'count'),
    
    # Regla 3 — peso total
    peso_total=('PESO_NETO_DESP', 'sum'),
    
    # Regla 1 — homogeneidad de material padre
    n_materiales_padre=('MATERIAL_PADRE', 'nunique'),
    
    # Regla 2 — homogeneidad de consignatario
    n_consignatarios=('CONSIGNATARIO', 'nunique'),
    
    # Regla 5 — diferencia de diámetros
    diff_diametro=('D_EXTERNO', lambda x: x.max() - x.min()),
    
    # Reglas 6 y 7 — tipo de embalaje (todos iguales dentro de la tarima)
    tipo_embalaje=('TIPO_EMBALAJE', 'first'),
    
    # Regla 6 — ancho máximo (relevante para OJO HORIZONTAL)
    ancho_max=('ANCHO_HIJO', 'max'),
    
    # Patrón identificado en exploración
    n_numeros_parte=('NUMERO_PARTE_HIJO', 'nunique'),
    
    # Planta (afecta límite de ancho en regla 6)
    D_UBICACION=('D_UBICACION', 'first'),

).reset_index()

print(f"Vista de tarimas: {vista_tarimas.shape[0]:,} filas × {vista_tarimas.shape[1]} columnas")
print(f"\nDistribución de DESARMA_TARIMA:")
print(vista_tarimas['DESARMA_TARIMA'].value_counts())
print(f"\nPorcentaje de tarimas desarmadas: {vista_tarimas['DESARMA_TARIMA'].mean()*100:.1f}%")

Vista de tarimas: 9,817 filas × 11 columnas

Distribución de DESARMA_TARIMA:
DESARMA_TARIMA
0    7788
1    2029
Name: count, dtype: int64

Porcentaje de tarimas desarmadas: 20.7%


In [4]:
print("=== Valores nulos por columna ===")
print(vista_tarimas.isna().sum())

print("\n=== Estadísticas generales ===")
print(vista_tarimas.describe().round(2))

=== Valores nulos por columna ===
ARMA_TARIMA           0
DESARMA_TARIMA        0
n_cintas              0
peso_total            0
n_materiales_padre    0
n_consignatarios      0
diff_diametro         0
tipo_embalaje         0
ancho_max             0
n_numeros_parte       0
D_UBICACION           0
dtype: int64

=== Estadísticas generales ===
       DESARMA_TARIMA  n_cintas  peso_total  n_materiales_padre  \
count         9817.00   9817.00     9817.00             9817.00   
mean             0.21      2.83        6.82                1.01   
std              0.40      1.47        5.87                0.10   
min              0.00      1.00        0.12                1.00   
25%              0.00      2.00        2.25                1.00   
50%              0.00      2.00        3.81                1.00   
75%              0.00      3.00       10.10                1.00   
max              1.00     16.00       40.60                4.00   

       n_consignatarios  diff_diametro  ancho_max  n_

In [5]:
vista_tarimas.to_csv('../outputs/02_vista_tarimas.csv', index=False)

print("Archivo guardado: outputs/02_vista_tarimas.csv")
print(f"Dimensiones finales: {vista_tarimas.shape[0]:,} filas × {vista_tarimas.shape[1]} columnas")

Archivo guardado: outputs/02_vista_tarimas.csv
Dimensiones finales: 9,817 filas × 11 columnas
